# Extracting and Visualizing Stock Data
## Tesla and GameStop — Final Assignment

**Objective:** Retrieve historical stock prices with `yfinance`, scrape quarterly revenue data with `BeautifulSoup`, and compare stock prices and revenue using interactive Plotly dashboards.

**Questions covered:** Q1–Q6.  
**Question 7:** Upload this completed notebook to a GitHub repository and submit the repository URL.

> Run the notebook from top to bottom in an internet-enabled Jupyter environment. The stock and revenue data are fetched from external sources, so the data cells require internet access.

## 1. Install and import the required libraries

In [ ]:
# Run this cell if the libraries are not already installed.
%pip install -q yfinance pandas requests beautifulsoup4 html5lib plotly

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Define the graphing function

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    """Plot historical share price and quarterly revenue in one dashboard."""

    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()

    stock_data["Date"] = pd.to_datetime(stock_data["Date"])
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"])

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.15
    )

    fig.add_trace(
        go.Scatter(
            x=stock_data["Date"],
            y=pd.to_numeric(stock_data["Close"], errors="coerce"),
            name="Share Price",
            mode="lines"
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=revenue_data["Date"],
            y=pd.to_numeric(revenue_data["Revenue"], errors="coerce"),
            name="Revenue",
            mode="lines+markers"
        ),
        row=2,
        col=1
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)

    fig.update_layout(
        title=f"{stock} Stock Price and Revenue Dashboard",
        height=850,
        showlegend=False,
        template="plotly_white",
        xaxis_rangeslider_visible=True
    )

    fig.show()

## Question 1 — Extract Tesla stock data using yfinance

Create a ticker object for Tesla (`TSLA`), retrieve the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

tesla_data.reset_index(inplace=True)
tesla_data.head(5)

## Question 2 — Extract Tesla revenue data using web scraping

Download the Tesla revenue webpage, parse its HTML using BeautifulSoup, locate the historical revenue table, clean the values, and display the last five rows.

In [ ]:
def extract_revenue_data(url):
    """Scrape a Date/Revenue table from a financial-history webpage."""

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    html_data = response.text
    soup = BeautifulSoup(html_data, "html5lib")

    # First try the historical table commonly used by Macrotrends.
    table = soup.find("table", id="historical_data_table")

    if table is not None:
        revenue_df = pd.read_html(str(table))[0]
    else:
        # Fallback: search all HTML tables for Date and Revenue columns.
        tables = pd.read_html(html_data)
        revenue_df = None

        for candidate in tables:
            candidate.columns = [str(col).strip() for col in candidate.columns]
            normalized = [col.lower() for col in candidate.columns]

            date_col = next((c for c in candidate.columns if c.lower() == "date"), None)
            revenue_col = next((c for c in candidate.columns if c.lower() == "revenue"), None)

            if date_col is not None and revenue_col is not None:
                revenue_df = candidate[[date_col, revenue_col]].copy()
                revenue_df.columns = ["Date", "Revenue"]
                break

        if revenue_df is None:
            raise ValueError(
                "A Date/Revenue table was not found. The source website may have "
                "changed its HTML or blocked automated requests."
            )

    revenue_df.columns = [str(col).strip() for col in revenue_df.columns]

    # Keep only the Date and Revenue columns.
    date_col = next(col for col in revenue_df.columns if col.lower() == "date")
    revenue_col = next(col for col in revenue_df.columns if col.lower() == "revenue")
    revenue_df = revenue_df[[date_col, revenue_col]].copy()
    revenue_df.columns = ["Date", "Revenue"]

    # Remove currency symbols and thousands separators.
    revenue_df["Revenue"] = (
        revenue_df["Revenue"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    revenue_df["Revenue"] = pd.to_numeric(revenue_df["Revenue"], errors="coerce")
    revenue_df["Date"] = pd.to_datetime(revenue_df["Date"], errors="coerce")

    revenue_df.dropna(subset=["Date", "Revenue"], inplace=True)
    revenue_df.sort_values("Date", inplace=True)
    revenue_df.reset_index(drop=True, inplace=True)

    return revenue_df


tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
tesla_revenue = extract_revenue_data(tesla_revenue_url)

tesla_revenue.tail(5)

## Question 3 — Extract GameStop stock data using yfinance

Create a ticker object for GameStop (`GME`), retrieve the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")

gme_data.reset_index(inplace=True)
gme_data.head(5)

## Question 4 — Extract GameStop revenue data using web scraping

Retrieve the GameStop revenue webpage, parse the HTML, clean the revenue values, and display the last five rows.

In [ ]:
gme_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
gme_revenue = extract_revenue_data(gme_revenue_url)

gme_revenue.tail(5)

## Question 5 — Plot the Tesla stock graph

Use the `make_graph()` function to display Tesla's historical share price and revenue.

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 — Plot the GameStop stock graph

Use the `make_graph()` function to display GameStop's historical share price and revenue.

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Final checklist

- [ ] Q1: `tesla_data.head(5)` is displayed.
- [ ] Q2: `tesla_revenue.tail(5)` is displayed.
- [ ] Q3: `gme_data.head(5)` is displayed.
- [ ] Q4: `gme_revenue.tail(5)` is displayed.
- [ ] Q5: Tesla dashboard is displayed.
- [ ] Q6: GameStop dashboard is displayed.
- [ ] Q7: Notebook is uploaded to GitHub and the repository URL is submitted.

**Submission tip:** Save the notebook after running every cell. Capture screenshots that show both the question heading/code and its resulting output, as requested by the assignment. Upload the corresponding screenshots to each question's upload field. For Q7, make sure the notebook is committed to a public or instructor-accessible GitHub repository.